In [19]:
import xarray as xr
import scipy.signal as signal
import pandas as pd
import glob


data_dir = '/home/lukeew/Local_work/data/01_SLW_xgb'
reference_mask_dir = data_dir + '/CHCH_v2_reference/processed{}/'

output_dir = data_dir + '/CHCH_v2_reference/'


In [3]:
date = '20210519'

infile = reference_mask_dir.format(date[:6]) + '/{}.nc'.format(date)

ds = xr.open_dataset(infile)
ds

<xarray.Dataset>
Dimensions:      (time: 288, level: 300)
Coordinates:
  * time         (time) datetime64[ns] 2021-05-19T00:02:29.999991 ... 2021-05...
    zfull        (level) float64 ...
Dimensions without coordinates: level
Data variables:
    backscatter  (time, level) float64 ...
    cloud_mask   (time, level) int8 ...
    temp         (time, level) float64 ...
    vdr          (time, level) float64 ...
    vdr_cm       (time, level) float64 ...
    wlwc         (time, level) int8 ...
    slwc         (time, level) int8 ...
    slcc         (time, level) int8 ...
    ic           (time, level) int8 ...
    undet        (time, level) int8 ...
    u_nan        (time, level) int8 ...
    unknown      (time, level) int8 ...

In [23]:
pd.read_csv('/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/peaks_training_df.csv')

,Unnamed: 0,date,peak_time_idx,peak_alt_idx,peak_magnitude,peak_width,peak_width_height,peak_prominence,peak_alt,num_peaks,peak_order,peak_temp,wlwc_target,slwc_target,slcc_target,ic_target,undet_target,unan_target
0,0,20210506,156,21,0.000022,1.208251,0.000011,0.000022,1075.0,1,0,NaN,0,0,0,0,0,0
1,1,20210506,161,21,0.000040,1.120655,0.000020,0.000040,1075.0,1,0,NaN,0,0,0,0,0,0
2,2,20210506,163,21,0.000101,1.028740,0.000051,0.000101,1075.0,1,0,NaN,0,0,0,0,0,0
3,3,20210506,164,20,0.000149,1.343672,0.000074,0.000148,1025.0,1,0,NaN,0,0,0,0,0,0
4,4,20210506,165,20,0.000207,1.391306,0.000104,0.000207,1025.0,1,0,NaN,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
42386,42386,20220120,284,63,0.000036,1.655835,0.000018,0.000036,3175.0,1,0,-4.586685,0,0,0,0,0,1
42387,42387,20220120,286,228,0.000026,3.522913,0.000013,0.000026,11425.0,1,0,-54.901438,0,0,0,1,0,0
42388,42388,20220121,35,53,0.000070,1.861522,0.000035,0.000070,2675.0,1,0,-0.646539,0,1,1,0,0,0
42389,42389,20220121,36,52,0.000105,1.742200,0.000052,0.000105,2625.0,1,0,-0.414722,0,0,1,0,0,0


In [17]:
backscatter = ds.backscatter.values 
altitude = ds.zfull.values
temperature = ds.temp.values

wlwc_target = ds.wlwc.values
slwc_target = ds.slwc.values
slcc_target = ds.slcc.values
ic_target = ds.ic.values
undet_target = ds.undet.values # Undetermined by reference mask
unan_target = ds.u_nan.values # Unknown due to missing data

magnitude_threshold = 2e-5
width_threshold = 1

date_peak_dict = []

for time_idx, profile in enumerate(backscatter):
    
    ### For each vertical profile, gets peak properties
    peaks, properties = signal.find_peaks(profile,height=magnitude_threshold, width=width_threshold, rel_height=0.5, prominence=(None,None))
    
    for peak_idx, peak_alt_idx in enumerate(peaks):
        
        peak_dict = {
            # Meta data
            'date' : date,
            'peak_time_idx' : time_idx,
            'peak_alt_idx' : peak_alt_idx,
            
            # Backscatter properties
            'peak_magnitude' : properties['peak_heights'][peak_idx],
            'peak_width' : properties['widths'][peak_idx],
            'peak_width_height' : properties['width_heights'][peak_idx],
            'peak_prominence' : properties['prominences'][peak_idx],
            'peak_alt' : altitude[peak_alt_idx],
            'num_peaks' : len(peaks),
            'peak_order' : peak_idx,
            
            # AMPS temperature
            'peak_temp' : temperature[time_idx, peak_alt_idx],
                        
            # Reference mask target            
            'wlwc_target' : wlwc_target[time_idx,peak_alt_idx],
            'slwc_target' : slwc_target[time_idx,peak_alt_idx],
            'slcc_target' : slcc_target[time_idx,peak_alt_idx],
            'ic_target' : ic_target[time_idx,peak_alt_idx],
            'undet_target' : undet_target[time_idx,peak_alt_idx],
            'unan_target' : unan_target[time_idx,peak_alt_idx],
                        
            }
        
        date_peak_dict.append(peak_dict)
    

In [15]:
peaks_df = pd.DataFrame.from_dict(date_peak_dict)
peaks_df

,date,peak_time_idx,peak_alt_idx,peak_magnitude,peak_width,peak_width_height,peak_prominence,peak_alt,num_peaks,peak_order,peak_temp,wlwc_target,slwc_target,slcc_target,ic_target,undet_target,unan_target
0,20210519,0,54,0.000183,2.470348,0.000092,0.000182,2725.0,2,0,-6.592864581139218,0,0,1,0,0,0
1,20210519,0,76,0.000068,1.550559,0.000034,0.000067,3825.0,2,1,-11.653900658450432,0,1,1,0,0,0
2,20210519,1,54,0.000272,2.031397,0.000136,0.000272,2725.0,2,0,-6.517793583903028,0,1,1,0,0,0
3,20210519,1,75,0.000027,2.013853,0.000013,0.000027,3775.0,2,1,-11.359696516954934,0,0,0,1,0,0
4,20210519,2,55,0.000041,2.633715,0.000021,0.000040,2775.0,2,0,-6.7768540416131176,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,20210519,182,3,0.000416,1.865601,0.000209,0.000412,175.0,1,0,12.774839763168515,1,0,0,0,0,0
118,20210519,183,3,0.000835,1.864364,0.000419,0.000832,175.0,1,0,12.754735095223753,1,0,0,0,0,0
119,20210519,184,4,0.000507,1.353679,0.000255,0.000504,225.0,1,0,12.807295536293264,1,0,0,0,0,0
120,20210519,185,4,0.000032,1.823729,0.000017,0.000029,225.0,1,0,12.789211163364005,1,0,0,0,0,0


In [22]:
# dates = [file[-11:-3].replace('-','') for file in sorted(glob.glob(reference_mask_dir.format('*')+'/*.nc'))]
# dates
sorted(glob.glob(reference_mask_dir.format('*')+'/*.nc'))

['/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210506.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210507.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210508.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210509.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210510.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210511.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210512.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210513.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210514.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210515.nc',
 '/home/lukeew/Local_work/data/01_SLW_xgb/CHCH_v2_reference/processed202105/20210516.nc',
 '/home/lu